# Semana 13 — Chunking e o Primeiro RAG Pipeline
**Fase 3: Advanced RAG, Search e Reranking** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_13_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-13-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-13-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import numpy as np, re, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

## Passo 1 — Um documento longo
Juntamos as políticas em um "Manual do Colaborador" em Markdown. É o tipo de fonte que um RAG corporativo recebe.

In [ ]:
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
manual = manual_demo(docs)
perguntas = [c for c in consultas if c["resposta"] in manual]
print(len(manual), "caracteres |", len(perguntas), "perguntas com resposta no manual\n"); print(manual[:500])

## Passo 2 — Chunking de tamanho fixo
Janelas de `tamanho` caracteres, avançando `tamanho - sobreposicao` a cada passo.

In [ ]:
def chunk_fixo(texto, tamanho=250, sobreposicao=40):
    passo = tamanho - sobreposicao
    return [texto[i:i + tamanho] for i in range(0, len(texto), passo)]

c_fixo = chunk_fixo(manual); print(len(c_fixo), "chunks\n---\n" + c_fixo[2])

Repare onde o chunk começa e termina.

## Passo 3 — Chunking recursivo por estrutura
Tente separar pelo separador mais forte (`"\n## "`); se um pedaço ainda for maior que `tamanho`, desça para o próximo separador (`"\n\n"`, depois `". "`). Pedaços vizinhos pequenos são reagrupados.

In [ ]:
def chunk_recursivo(texto, tamanho=250, seps=("\n## ", "\n\n", ". ")):
    if len(texto) <= tamanho or not seps:
        return [texto.strip()] if texto.strip() else []
    saida, atual = [], ""
    for parte in texto.split(seps[0]):
        candidato = atual + seps[0] + parte if atual else parte
        if len(candidato) <= tamanho:
            atual = candidato
        else:
            if atual: saida.append(atual)
            atual = ""
            if len(parte) > tamanho: saida.extend(chunk_recursivo(parte, tamanho, seps[1:]))
            else: atual = parte
    if atual: saida.append(atual)
    return [c.strip() for c in saida if c.strip()]

c_rec = chunk_recursivo(manual); print(len(c_rec), "chunks\n---\n" + c_rec[2])

## Passo 4 — Chunking semântico
Divida em frases, gere embeddings e calcule a similaridade entre frases consecutivas. Abra um novo chunk quando a similaridade cair abaixo do percentil 30.

In [ ]:
emb_model = SentenceTransformer(MODELO_EMB)
embed = lambda textos: emb_model.encode(list(textos), normalize_embeddings=True, convert_to_numpy=True)

def chunk_semantico(texto, percentil=30):
    frases = [f.strip() for f in re.split(r"(?<=[.!?])\s+|\n+", texto) if f.strip() and not f.startswith("#")]
    E = embed(frases)
    sims = np.sum(E[:-1] * E[1:], axis=1)                 # similaridade entre frase i e i+1
    limiar = np.percentile(sims, percentil)
    chunks, atual = [], [frases[0]]
    for frase, s in zip(frases[1:], sims):
        if s < limiar:
            chunks.append(" ".join(atual)); atual = []
        atual.append(frase)
    chunks.append(" ".join(atual))
    return chunks

c_sem = chunk_semantico(manual); print(len(c_sem), "chunks\n---\n" + c_sem[2])

## Passo 5 — Qual chunking preserva as respostas?
Métrica simples e reveladora: para cada pergunta, a frase-resposta aparece **inteira** em algum dos `k` chunks recuperados?

In [ ]:
def avaliar_chunking(chunks, k=2):
    E = embed(chunks); acertos = 0
    for c in perguntas:
        top = np.argsort(-(E @ embed([c["q"]])[0]))[:k]
        acertos += any(c["resposta"] in chunks[i] for i in top)
    return acertos / len(perguntas)

for nome, ch in [("fixo", c_fixo), ("recursivo", c_rec), ("semântico", c_sem)]:
    print(f"{nome:10s} | {len(ch):3d} chunks | tamanho médio {np.mean([len(x) for x in ch]):5.0f} | resposta inteira no top-2: {avaliar_chunking(ch):.2f}")

## Passo 6 — Carregar o LLM

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()

def gerar(mensagens, max_new_tokens=120):
    prompt = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    ent = tok(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        out = llm.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(gerar([{"role": "user", "content": "Responda em uma frase: o que é RAG?"}]))

## Passo 7 — Sem contexto: o modelo inventa

In [ ]:
pergunta = "Qual o limite diário de alimentação em viagens nacionais na nossa empresa?"
print(gerar([{"role": "user", "content": pergunta}]))

O modelo não tem como saber. O que ele respondeu?

## Passo 8 — O pipeline RAG
Escreva `montar_prompt`: uma mensagem de sistema com as regras de grounding e uma mensagem de usuário com os chunks numerados (`[1] ...`) seguidos da pergunta.

In [ ]:
chunks = c_rec; E_chunks = embed(chunks)

def recuperar(q, k=3):
    idx = np.argsort(-(E_chunks @ embed([q])[0]))[:k]
    return [chunks[i] for i in idx]

def montar_prompt(q, trechos):
    sistema = ("Você é um assistente interno. Responda SOMENTE com base no CONTEXTO fornecido. "
               "Se a resposta não estiver no contexto, responda exatamente: 'Não encontrei essa informação.' "
               "Seja direto e cite o número do trecho usado entre colchetes, por exemplo [1].")
    contexto = "\n".join(f"[{i + 1}] {t}" for i, t in enumerate(trechos))
    return [{"role": "system", "content": sistema},
            {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA: {q}"}]

def rag(q, k=3, verbose=False):
    trechos = recuperar(q, k); msgs = montar_prompt(q, trechos)
    if verbose: print(msgs[1]["content"], "\n" + "-" * 60)
    return gerar(msgs)

print(rag(pergunta, verbose=True))

## Passo 9 — Pergunta sem resposta na base

In [ ]:
for q in ["Qual é a política de estacionamento da sede?", "Quem ganhou a Copa de 2002?"]:
    print(f"{q}\n   -> {rag(q)}\n")

O modelo recusou ou inventou? Se inventou, ajuste o prompt de sistema e teste de novo.

## Passo 10 — O custo de `k`

In [ ]:
for k in (1, 3, 6):
    msgs = montar_prompt(pergunta, recuperar(pergunta, k))
    texto_prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    n_tokens = len(tok(texto_prompt, add_special_tokens=False)["input_ids"])
    print(f"k={k} | tokens de entrada: {n_tokens}")

## Teste Prático de Entrevista — Chunking com Overlap e Proteção de Bordas
Um entrevistador pede: *"Implemente `chunk_words(text, chunk_size, overlap)` dividindo o texto em janelas de palavras com overlap. Se `chunk_size <= 0` ou `overlap < 0` ou `overlap >= chunk_size`, lance `ValueError` para impedir loop infinito."* 

In [ ]:
def chunk_words(text, chunk_size, overlap):
    """Divide texto em chunks de palavras com overlap garantido."""
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Parametros invalidos: exija 0 <= overlap < chunk_size")
    words = text.split()
    if not words:
        return []
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start : start + chunk_size]
        chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


chunks = chunk_words("um dois tres quatro cinco seis", chunk_size=3, overlap=1)
assert chunks == ["um dois tres", "tres quatro cinco", "cinco seis"], f"Veio {chunks}"
try:
    chunk_words("a b c", chunk_size=2, overlap=2)
    raise AssertionError("Deveria rejeitar overlap >= chunk_size")
except ValueError:
    pass
print("OK! Chunks:", chunks)

## Fechamento
Registre em `docs/` a tabela do Passo 5 e responda: qual estratégia você adotaria para os documentos da sua empresa e por quê?